# Chapter 3 standalone: the golden dataset, end to end

*Google Gemini edition*

**Author: Imran Ahmad** · Companion notebook for *Building Reliable AI-Assisted Software Systems* (Packt). All characters, companies, incidents, and data are fictional.

## Objectives

This notebook walks the chapter's central artifact from bytes on disk to the team's first honest measurement. By the last cell you will have:

- verified a committed golden dataset against its hash manifest, the way CI will in Chapter 5
- validated all 150 records against the `GoldenExample` schema and their canonical category counts
- met the eleven phrasings of one customer question, including the incident replay `golden_0047`
- exercised a deterministic criterion against its own calibration pair
- reproduced the honest number from the committed hand-graded verdicts: 108 of 150, 72 percent

Everything runs offline from committed artifacts. No API key, no network, no randomness that matters: run it twice and every number matches.

The chapter's opening argument in one image: correctness is a profile, not a score. One response can average well and still fail the axis that matters.

<img src="assets/fig_3_1_correctness_profile_300dpi.png" width="700" alt="correctness profile"/>

Six healthy dimensions, one collapsed safety axis, and a blended average that says ship it. The dataset this notebook loads exists to catch exactly that response.

### Provider edition: Google Gemini

This edition adds one optional live cell that sends a single golden case to a Gemini model and runs the C3 check on the reply. Requirements beyond the base bundle: `google-genai` (see `requirements-providers.txt`). Set `GEMINI_API_KEY` to enable the live probe. **Simulation Mode is the canonical path**: with no key, every number in this notebook still reproduces from the committed artifacts, and the live cell skips itself.

In [1]:
# Chapter 3 - Defining Correctness and Building Golden Datasets
# Author: Imran Ahmad
# Standalone companion notebook (core). Runs fully offline; every number is canon.
from __future__ import annotations

import csv
import hashlib
import json
import random
from collections import Counter
from pathlib import Path

from renderloft_mock import (
    CANON,
    CATEGORIES,
    CRITERIA,
    DATA_DIR,
    SEED,
    GoldenExample,
    MONEY_BACK_PHRASINGS,
)

random.seed(SEED)

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("setup complete: modules loaded, seed", SEED)

setup complete: modules loaded, seed 42


## The dataset is code: integrity and schema

A golden dataset earns trust the way source code does: it has a version, a hash, and a review trail. The first two cells refuse to work with anything else. The manifest check catches silent edits, and the schema pass catches records that drifted from the contract.

<img src="assets/fig_3_6_lifecycle_loop_300dpi.png" width="700" alt="dataset lifecycle"/>

The loop above is the dataset's whole life: add, review, version, evaluate, monitor, retire. This notebook enters at the evaluate stage, with version 1.0.0 already on disk.

In [2]:
# 1. Integrity: the committed dataset matches its manifest
manifest = json.loads((DATA_DIR / "golden_MANIFEST.json").read_text())
golden_path = DATA_DIR / "golden_v1.0.jsonl"
assert sha256(golden_path) == manifest["sha256"], "dataset drifted from manifest"
print(f"golden v{manifest['version']}: hash verified, "
      f"{manifest['case_count']} cases expected")

golden v1.0.0: hash verified, 150 cases expected


In [3]:
# 2. Schema validation + stratified counts
cases = [GoldenExample.model_validate_json(line)
         for line in golden_path.read_text().splitlines() if line.strip()]
assert len(cases) == int(CANON["GOLDEN_CASES"])
counts = Counter(case.tags[0] for case in cases)
assert dict(counts) == {k: v["cases"] for k, v in CATEGORIES.items()}
print(f"{len(cases)} records schema-validated; per-category counts match canon:")
for cat, n in counts.items():
    print(f"   {cat:<24} {n}")

150 records schema-validated; per-category counts match canon:
   refund_policy            38
   product_howto            20
   account_access           20
   billing_licensing        30
   technical_support        27
   out_of_scope_adversarial 15


The counts are not the queue's proportions. Refund and adversarial cases are deliberately overweighted, because a golden dataset is a portfolio of the failures you refuse to ship, weighted by cost.

## Curating cases that reflect production

Two days of ticket traffic produced eleven distinct ways of asking one question. The pair to watch: `golden_0047` (annual license, six weeks after activation, must be refused) and `golden_0050` (freelancer on day 21, the guarantee applies). Similar phrasing, opposite correct answers, which is why every record carries its full context.

<img src="assets/fig_3_4_golden_anatomy_300dpi.png" width="700" alt="golden example anatomy"/>

Every field in the record above has a consumer: the context replays the incident's exact conditions, the rationale preserves why refusal is correct, and the tags drive the per-category scoring you will see below.

In [4]:
# 3. The eleven money-back phrasings
family = {c.id: c for c in cases if c.id in MONEY_BACK_PHRASINGS}
assert len(family) == int(CANON["PHRASING_COUNT"])
for cid in sorted(family):
    print(f"   {cid}: {family[cid].input[:58]}")
print(f"phrasing family: {len(family)} cases, incident replay = golden_0047")

   golden_0040: can I get my money back?
   golden_0041: I want a refund.
   golden_0042: how do I get refunded for my Studio subscription
   golden_0043: money back please - this isn't working for us
   golden_0044: is it possible to undo my purchase?
   golden_0045: hi, I'd like to return Renderloft Studio
   golden_0046: can you reverse the charge on my card from last month?
   golden_0047: We activated six weeks ago. Can we still get our money bac
   golden_0048: I bought this by mistake, can I be reimbursed?
   golden_0049: what happens if I'm not happy with the product? money back
   golden_0050: do I qualify to get my payment back after 3 weeks?
phrasing family: 11 cases, incident replay = golden_0047


## Rubrics and calibration pairs

Six criteria, C1 to C6, define correctness for this assistant. Three are ordinary code (C2, C3, C6) and three need a reader (C1, C4, C5), which is Chapter 4's business. The pyramid rule: push every check as far down as it will honestly go.

<img src="assets/fig_3_2_check_pyramid_300dpi.png" width="700" alt="check pyramid"/>

The cell below runs the chapter's one live check. A criterion ships with a pass example and a fail example, and a check that cannot fail its own fail example is a bug. C3's fail example smuggles in a 30-day guarantee that its context never granted, and the numeric scan catches it.

In [5]:
# 4. One deterministic criterion, exercised on its calibration pair
c3 = CRITERIA["C3"]
assert c3.check is not None
assert c3.check(c3.pass_example, c3.pass_example) is True
assert c3.check(c3.fail_example, c3.pass_example) is False
print("C3 calibration pair behaves: pass example passes, fail example fails")

C3 calibration pair behaves: pass example passes, fail example fails


## The honest number

The committed grade sheet holds two graders' verdicts on all 150 cases, produced over three afternoons. The aggregate hides the story, so the cell prints the slices too: the categories with money and safety attached score far below the headline.

In [6]:
# 5. The hand-graded verdicts reproduce the honest number
with (DATA_DIR / "grades_v1.0.csv").open() as fh:
    rows = list(csv.DictReader(fh))
passes = sum(row["verdict"] == "pass" for row in rows)
fails = sum(row["verdict"] == "fail" for row in rows)
rate = passes * 100 // len(rows)
assert (passes, fails, rate) == (int(CANON["PASS_COUNT"]),
                                 int(CANON["FAIL_COUNT"]),
                                 int(CANON["PASS_RATE_PCT"]))
print("per-category slice rates:")
for cat, spec in CATEGORIES.items():
    got = sum(r["verdict"] == "pass" for r in rows if r["category"] == cat)
    print(f"   {cat:<24} {got}/{spec['cases']:<3} = {got * 100 / spec['cases']:.1f}%")

per-category slice rates:
   refund_policy            24/38  = 63.2%
   billing_licensing        23/30  = 76.7%
   technical_support        21/27  = 77.8%
   account_access           16/20  = 80.0%
   product_howto            17/20  = 85.0%
   out_of_scope_adversarial 7/15  = 46.7%


Read the slices before the total: 46.7 percent on adversarial cases and 63.2 percent on refunds is the diagnosis, and the 85 percent on how-to questions is the anesthetic. The demo had felt like roughly 95.

In [7]:
# 6. Held-out split by ID arithmetic
held = [c.id for c in cases if int(c.id[-4:]) % 5 == 0]
print(f"held-out split: {len(cases) - len(held)} dev / {len(held)} held out "
      "(mechanical, so nobody curates an easy holdout)")

held-out split: 120 dev / 30 held out (mechanical, so nobody curates an easy holdout)


Thirty cases, every ID divisible by five, never used for prompt iteration. The split is ID arithmetic precisely so nobody curates an easy holdout.

In [8]:
# 7. The final printout
print(f"the count: {passes}/{len(rows)} = {rate}%")

the count: 108/150 = 72%


In [9]:
# Optional live probe (Gemini): one golden case, one reply, one deterministic check.
# The canonical numbers above come from the committed snapshot in BOTH modes.
import os

case = next(c for c in cases if c.id == "golden_0040")
if os.getenv("GEMINI_API_KEY"):
    from google import genai

    llm = genai.Client()  # reads GEMINI_API_KEY from the environment
    reply = llm.models.generate_content(
        model="gemini-2.5-flash",  # check llm.models.list() for newer ids
        contents=("You are a support assistant. Answer strictly from the "
                  "provided policy context; invent no figures.\n\n"
                  f"{case.context}\n\nCustomer: {case.input}"),
    )
    text = reply.text or "(empty reply)"
    print("live reply:", text[:200])
    print("C3 (no invented figures) on the live reply:",
          CRITERIA["C3"].check(text, case.context or ""))
else:
    print("SIMULATION MODE: GEMINI_API_KEY not set; live probe skipped.")
    print("All canonical numbers above came from the committed snapshot.")

SIMULATION MODE: GEMINI_API_KEY not set; live probe skipped.
All canonical numbers above came from the committed snapshot.


## Summary

The working rule this notebook leaves behind: define correctness as data, then argue with the data. A hash-verified, schema-validated, version-numbered dataset turned "the assistant seems fine" into 108 of 150, and the slice table turned that number into a to-do list.

Hand-grading 150 cases cost two senior people three afternoons, and that bill caps how often the team can afford the truth. The batch grader in this bundle is still a stub that raises `NotImplementedError("Chapter 4 builds the judge.")`. Chapter 4 pays that debt: an automated judge for C1, C4, and C5, calibrated against the very verdicts this notebook just loaded.

## Exercises

1. **Reflection:** which of your product's correctness dimensions are gates (any failure fails the case) and which are graded? Name a fail example for each, or admit the dimension is not defined yet.
2. **Application:** add one new golden case to a copy of `golden_v1.0.jsonl` (input, context, criteria, rationale, source, tags), rerun this notebook, and watch the manifest check fail. Version the change properly and regenerate the manifest.
3. **Discussion:** the held-out rule here is mechanical. Argue for or against letting the team hand-pick the holdout instead, then check your argument against what happens to a benchmark whose test set everyone optimizes on.